## Title: DSC355 USL Championship 2025 — Feature Engineering
# Author: James Sneddon
# Date: 12 April 2026
# Description: Feature engineering on USL Championship 2025 match-level team statistics.

In [1]:
import pandas as pd
import numpy as np
import re
from sklearn.preprocessing import StandardScaler 
from sklearn.preprocessing import MinMaxScaler

In [2]:
df = pd.read_csv('data/USL_Championship_2025_Team_Stats.csv')
df['Date'] = pd.to_datetime(df['Date'])
df.head()

,Team,Date,Match,Competition,Duration,Scheme,Goals,xG,Shots,Shots on target,...,Throw ins,Accurate throw ins,Throw in accuracy %,Goal kicks,Match tempo,Average passes per possession,Long pass %,Average shot distance,Average pass length,PPDA
0,Birmingham Legion,2025-10-25,Charleston Battery - Birmingham Legion 2:1,United States. USL Championship,98,5-3-2 (51.4%),1,0.22,3.0,3.0,...,20.0,19.0,95.00,11.0,14.54,3.11,16.97,26.59,19.27,28.92
1,Birmingham Legion,2025-10-19,Birmingham Legion - Miami 2:3,United States. USL Championship,105,3-5-2 (83.46%),2,2.71,14.0,4.0,...,25.0,23.0,92.00,8.0,16.30,4.89,7.49,17.63,19.96,8.56
2,Birmingham Legion,2025-10-04,Loudoun United - Birmingham Legion 0:1,United States. USL Championship,100,3-5-2 (100.0%),1,1.19,6.0,2.0,...,21.0,16.0,76.19,9.0,14.71,2.89,21.91,19.80,22.49,17.95
3,Birmingham Legion,2025-10-02,Birmingham Legion - North Carolina FC 1:1,United States. USL Championship,101,5-3-2 (100.0%),1,1.00,11.0,3.0,...,28.0,26.0,92.86,5.0,16.61,5.91,7.18,19.98,18.30,13.11
4,Birmingham Legion,2025-09-28,San Antonio - Birmingham Legion 0:0,United States. USL Championship,105,5-3-2 (80.27%),0,0.25,6.0,2.0,...,13.0,11.0,84.62,12.0,15.77,3.07,14.18,19.90,20.88,12.44


## 1. Missing Value Check

In [3]:
missing = df.isnull().sum()
print(missing)

Team                             0
Date                             0
Match                            0
Competition                      0
Duration                         0
                                ..
Average passes per possession    0
Long pass %                      0
Average shot distance            0
Average pass length              0
PPDA                             0
Length: 109, dtype: int64


## 2. Playoff / Extra-Time
The EDA noted that `Duration` occasionally exceeds 105 minutes (max = 194), which indicates extra time played in knockout/playoff matches. We flag these separately since their statistics are not comparable to 90-minute league matches.

In [4]:
# Threshold: > 105 min = extra time was played, binary var to allow for future sorting of playoff matches (Still considering whether it's better to just exempt playoff matches at large)
df['Is_Extra_Time'] = (df['Duration'] > 105).astype(int)

## 3. Extracting Match Outcome Features from the Match Column
3 pts win, 1 pt draw, 0 pts loss, can be used for a cumulative var that is explanatory

In [5]:
def get_features(row):
    m = re.search(r'(\d+):(\d+)', row['Match'])
    hs, as_ = int(m.group(1)), int(m.group(2))
    home = re.split(r'\s*\(?\w\)?\s*\d+:\d+', row['Match'])[0].split(' - ')[0].strip()
    is_home = row['Team'] == home
    ms, os_ = (hs, as_) if is_home else (as_, hs)
    r = 'W' if ms > os_ else ('L' if ms < os_ else 'D')
    return pd.Series([int(is_home), r, 3 if r=='W' else 1 if r=='D' else 0],
                     index=['Is_Home', 'Result', 'Points'])

## 4. Formation Simplification
Turns formation into a clean, categorical var

In [6]:
# Extract formation pattern like '4-3-3', '5-3-2', '4-2-3-1'
df['Formation'] = df['Scheme'].str.extract(r'^(\d[\d-]+\d)')

# Count how many distinct formations appear
print(f'\nUnique formations: {df["Formation"].nunique()}')

formation_counts = df['Formation'].value_counts()
rare_formations = formation_counts[formation_counts < 5].index
df['Formation_Clean'] = df['Formation'].where(~df['Formation'].isin(rare_formations), other='Other')

print('Formation_Clean distribution:')
print(df['Formation_Clean'].value_counts())


Unique formations: 17
Formation_Clean distribution:
Formation_Clean
4-2-3-1    190
4-4-2      106
3-4-3      100
5-4-1       81
4-1-4-1     50
5-3-2       47
3-4-2-1     39
3-4-1-2     39
4-3-3       30
3-5-2       17
4-3-1-2     17
4-1-3-2     16
Other        8
4-4-1-1      6
Name: count, dtype: int64


## 5. Season Phase Binning
Binning match dates into Early / Mid / Late season, could be useful for future analyses

In [7]:
# Compute tertile cut-points from the regular season dates only
regular = df[df['Date'] < '2025-11-01']['Date'].astype(np.int64)
t1 = pd.Timestamp(regular.quantile(0.33).astype(np.int64))
t2 = pd.Timestamp(regular.quantile(0.66).astype(np.int64))

def season_phase(date):
    if date >= pd.Timestamp('2025-11-01'):
        return 'Playoffs'
    elif date <= t1:
        return 'Early'
    elif date <= t2:
        return 'Mid'
    else:
        return 'Late'

df['Season_Phase'] = df['Date'].apply(season_phase)
print(f'Tertile cut-points — Early ends: {t1.date()}, Mid ends: {t2.date()}')
print('\nSeason_Phase distribution:')
print(df['Season_Phase'].value_counts())

Tertile cut-points — Early ends: 2025-05-25, Mid ends: 2025-08-24

Season_Phase distribution:
Season_Phase
Mid         250
Early       240
Late        226
Playoffs     30
Name: count, dtype: int64


## 6. Binning PPDA

In [8]:
q1, q2, q3 = df['PPDA'].quantile([0.25, 0.50, 0.75])
print(f'PPDA quartiles  Q1={q1:.2f}  Q2={q2:.2f}  Q3={q3:.2f}')
print(f'PPDA range      min={df["PPDA"].min():.2f}  max={df["PPDA"].max():.2f}')

df['Pressing_Intensity'] = pd.cut(
    df['PPDA'],
    bins=[0, q1, q2, q3, df['PPDA'].max() + 1],
    labels=['High Press', 'Moderate-High', 'Moderate-Low', 'Low Press'],
    right=False
)

print('\nPressing_Intensity distribution:')
print(df['Pressing_Intensity'].value_counts().sort_index())

PPDA quartiles  Q1=8.10  Q2=10.46  Q3=12.91
PPDA range      min=3.81  max=44.86

Pressing_Intensity distribution:
Pressing_Intensity
High Press       186
Moderate-High    187
Moderate-Low     186
Low Press        187
Name: count, dtype: int64


## 7.Binning Possession %
Grouping possession into Low / Medium / High

In [9]:
# Fixed thresholds: < 45% = Low, 45–55% = Medium, > 55% = High
df['Possession_Tier'] = pd.cut(
    df['Possession %'],
    bins=[0, 45, 55, 100],
    labels=['Low', 'Medium', 'High']
)

print('Possession_Tier distribution:')
print(df['Possession_Tier'].value_counts())

Possession_Tier distribution:
Possession_Tier
Medium    318
Low       214
High      214
Name: count, dtype: int64


## 9. Derived Ratio Features
Create useful columns that provide more info than the initial by itself

In [10]:
# 9a. xG Difference — how much a team over/underperformed their shot quality
df['xG_Diff'] = (df['Goals'] - df['xG']).round(3)

# 9b. Shot Conversion Rate — goals per shot on target (avoid divide-by-zero)
df['Shot_Conversion_Rate'] = np.where(
    df['Shots on target'] > 0,
    (df['Goals'] / df['Shots on target']).round(4),
    0
)

## 10. Normalization
Preparing for future regression 

-z-score
-MinMaxScaler (0–1)

In [11]:
# Columns selected for modeling — based onhigh correlation with Goals per EDA
model_features = [
    'xG', 'Shots', 'Shots on target', 'Shots on target %',
    'Possession %', 'Pass accuracy %', 'PPDA',
    'Touches in penalty area', 'Positional attacks',
    'Counterattacks', 'Deep completed passes',
    'Shots against on target', 'Defensive duels won %',
    'Interceptions', 'Clearances',
    'Average passes per possession', 'Long pass %',
    'Average shot distance', 'Match tempo',
    'xG_Diff', 'Shot_Conversion_Rate'
]

In [12]:
# Standard (z-score) normalization → _z suffix
scaler_z = StandardScaler()
z_scaled = scaler_z.fit_transform(df[model_features])
z_col_names = [f'{c}_z' for c in model_features]
df_z = pd.DataFrame(z_scaled, columns=z_col_names, index=df.index)
df = pd.concat([df, df_z], axis=1)

print('Z-score columns added (sample):')
df[z_col_names[:5]].describe().round(3)

Z-score columns added (sample):


,xG_z,Shots_z,Shots on target_z,Shots on target %_z,Possession %_z
count,746.000,746.000,746.000,746.000,746.000
mean,0.000,-0.000,0.000,0.000,0.000
std,1.001,1.001,1.001,1.001,1.001
min,-1.686,-2.230,-1.763,-2.075,-3.140
25%,-0.762,-0.579,-0.803,-0.633,-0.672
50%,-0.177,-0.107,-0.323,-0.152,0.000
75%,0.516,0.600,0.637,0.673,0.672
max,6.250,3.902,3.997,3.693,3.140


In [13]:
# Min-Max normalization → _norm suffix
scaler_mm = MinMaxScaler()
mm_scaled = scaler_mm.fit_transform(df[model_features])
mm_col_names = [f'{c}_norm' for c in model_features]
df_mm = pd.DataFrame(mm_scaled, columns=mm_col_names, index=df.index)
df = pd.concat([df, df_mm], axis=1)

print('Min-Max columns added (sample — all values in [0,1]):')
df[mm_col_names[:5]].describe().round(3)

Min-Max columns added (sample — all values in [0,1]):


,xG_norm,Shots_norm,Shots on target_norm,Shots on target %_norm,Possession %_norm
count,746.000,746.000,746.000,746.000,746.000
mean,0.212,0.364,0.306,0.360,0.500
std,0.126,0.163,0.174,0.173,0.159
min,0.000,0.000,0.000,0.000,0.000
25%,0.116,0.269,0.167,0.250,0.393
50%,0.190,0.346,0.250,0.333,0.500
75%,0.277,0.462,0.417,0.476,0.607
max,1.000,1.000,1.000,1.000,1.000


In [14]:
# Save engineered dataset for use in modeling notebook
out_path = 'data/USL_Championship_2025_Feature_Engineered.csv'
df.to_csv(out_path, index=False)